In [12]:
import os
import glob
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights

import re

# ==========================================
# 1. DATASET DEFINITION
# ==========================================
class TextCropDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_dir: str, transform=None, is_test: bool = False):
        self.df = df
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir / row["image_path"]
        
        # Загрузка изображения
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, row.get("image_path", idx)
        else:
            label = torch.tensor(row["is_180"], dtype=torch.float32)
            return image, label


# ==========================================
# 2. TRANSFORMS & AUGMENTATIONS
# ==========================================
def get_transforms(img_size=(64, 192)):
    """
    Размер (64, 192) хорошо подходит для большинства текстовых кропов,
    сохраняя вытянутое соотношение сторон.
    """
    train_transform = T.Compose([
        T.Resize(img_size),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        T.RandomGrayscale(p=0.1),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    val_test_transform = T.Compose([
        T.Resize(img_size),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    return train_transform, val_test_transform


# ==========================================
# 3. MODEL BUILDER
# ==========================================
def build_resnet_model(pretrained: bool = True) -> nn.Module:
    weights = ResNet18_Weights.DEFAULT if pretrained else None
    model = resnet18(weights=weights)
    
    # Заменяем финальный полносвязный слой на одно число (логит)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(in_features, 1)
    )
    return model


# ==========================================
# 4. BRIER SCORE EVALUATION
# ==========================================
def compute_brier_score(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    """Brier Score = Mean Squared Error между вероятностями и истинными классами."""
    return float(np.mean((y_prob - y_true) ** 2))


# ==========================================
# 5. TEMPERATURE SCALING FOR CALIBRATION
# ==========================================
class TemperatureScaler(nn.Module):
    """
    Пост-обработка логитов для калибровки вероятностей под Brier Score.
    Подбирает оптимальную температуру T на валидационной выборке.
    """
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

    def fit(self, val_logits: torch.Tensor, val_labels: torch.Tensor):
        self.to(val_logits.device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.LBFGS([self.temperature], lr=0.01, max_iter=50)

        def eval_fn():
            optimizer.zero_grad()
            loss = criterion(self.forward(val_logits), val_labels)
            loss.backward()
            return loss

        optimizer.step(eval_fn)
        print(f"Оптимальная температура калибровки (T): {self.temperature.item():.4f}")


# ==========================================
# 6. TRAINING & INFERENCE PIPELINE
# ==========================================
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(dataloader.dataset)


@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    all_logits = []
    all_labels = []

    for images, labels in dataloader:
        images = images.to(device)
        logits = model(images)
        all_logits.append(logits.cpu())
        all_labels.append(labels.unsqueeze(1))

    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    probs = torch.sigmoid(all_logits).numpy()

    brier = compute_brier_score(all_labels.numpy(), probs)
    return brier, all_logits, all_labels


@torch.no_grad()
def predict_test(model, scaler, dataloader, device, use_tta: bool = True):
    """
    Предсказание вероятностей с TTA и форматированием итогового image_id.
    """
    model.eval()
    all_paths = []
    all_probs = []

    for images, img_paths in dataloader:
        images = images.to(device)

        # Прямой проход
        logits = model(images)
        if scaler is not None:
            logits = scaler(logits)
        probs = torch.sigmoid(logits)

        # Test-Time Augmentation (TTA)
        if use_tta:
            images_rotated = torch.flip(images, dims=[2, 3])  # поворот на 180°
            logits_rot = model(images_rotated)
            if scaler is not None:
                logits_rot = scaler(logits_rot)
            probs_rot = torch.sigmoid(logits_rot)
            
            # Усреднение: p_final = (p_orig + (1 - p_rot)) / 2
            probs = (probs + (1.0 - probs_rot)) / 2.0

        probs_np = probs.cpu().numpy().flatten()
        all_paths.extend(img_paths)
        all_probs.extend(probs_np)

    # Формирование итоговых колонок image_id и p_180
    image_ids = [extract_image_id(path, i) for i, path in enumerate(all_paths)]

    return pd.DataFrame({
        "image_id": image_ids,
        "p_180": all_probs
    })

def extract_image_id(file_name: str, idx: int) -> str:
    """
    Преобразует имя файла или индекс в формат 'test_00000'.
    Например: 'crop_000012.jpg' -> 'test_00012', '123.png' -> 'test_00123'
    """
    stem = Path(file_name).stem
    numbers = re.findall(r"\d+", stem)
    if numbers:
        num = int(numbers[-1])
        return f"test_{num:05d}"
    return f"test_{idx:05d}"



In [8]:
# ==========================================
# 7. MAIN EXECUTION FLOW
# ==========================================

train_csv_path = "./data/synthetic_ocr_dataset/labels.csv"
data_dir = "./data/synthetic_ocr_dataset"
test_dir = "test/images"
output_sub_path = "submission.csv"
epochs = 20
batch_size = 64
lr = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")

# --- Подготовка данных ---
df_train_full = pd.read_csv(train_csv_path)
train_df, val_df = train_test_split(df_train_full, test_size=0.15, random_state=42, stratify=df_train_full["is_180"])

train_tf, val_tf = get_transforms(img_size=(64, 192))

train_ds = TextCropDataset(train_df, data_dir, transform=train_tf)
val_ds = TextCropDataset(val_df, data_dir, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# --- Обучение модели ---
model = build_resnet_model(pretrained=True).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

best_brier = float("inf")
best_model_weights = None



Используется устройство: cpu


In [9]:
print("\n--- Старт обучения ---")
for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_brier, _, _ = evaluate(model, val_loader, device)
    scheduler.step()

    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Val Brier Score: {val_brier:.5f}")

    if val_brier < best_brier:
        best_brier = val_brier
        best_model_weights = model.state_dict().copy()

print(f"\nЛучший Brier Score на валидации: {best_brier:.5f}")
model.load_state_dict(best_model_weights)


--- Старт обучения ---
Epoch 01/20 | Train Loss: 0.5728 | Val Brier Score: 0.11617
Epoch 02/20 | Train Loss: 0.1552 | Val Brier Score: 0.06470
Epoch 03/20 | Train Loss: 0.0456 | Val Brier Score: 0.04675
Epoch 04/20 | Train Loss: 0.0228 | Val Brier Score: 0.04923
Epoch 05/20 | Train Loss: 0.0208 | Val Brier Score: 0.04481
Epoch 06/20 | Train Loss: 0.0087 | Val Brier Score: 0.04104
Epoch 07/20 | Train Loss: 0.0129 | Val Brier Score: 0.03345
Epoch 08/20 | Train Loss: 0.0078 | Val Brier Score: 0.03837
Epoch 09/20 | Train Loss: 0.0061 | Val Brier Score: 0.04495
Epoch 10/20 | Train Loss: 0.0025 | Val Brier Score: 0.03691
Epoch 11/20 | Train Loss: 0.0023 | Val Brier Score: 0.03090
Epoch 12/20 | Train Loss: 0.0020 | Val Brier Score: 0.03005
Epoch 13/20 | Train Loss: 0.0013 | Val Brier Score: 0.02896
Epoch 14/20 | Train Loss: 0.0018 | Val Brier Score: 0.02613
Epoch 15/20 | Train Loss: 0.0019 | Val Brier Score: 0.03010
Epoch 16/20 | Train Loss: 0.0011 | Val Brier Score: 0.02890
Epoch 17/20 | Tr

<All keys matched successfully>

In [13]:
# --- Калибровка вероятностей (Temperature Scaling) ---
_, val_logits, val_labels = evaluate(model, val_loader, device)
scaler = TemperatureScaler()
scaler.fit(val_logits.to(device), val_labels.to(device))

# Оценка Brier Score после калибровки
with torch.no_grad():
    calibrated_logits = scaler(val_logits.to(device))
    calibrated_probs = torch.sigmoid(calibrated_logits).cpu().numpy()
    
calibrated_brier = compute_brier_score(val_labels.numpy(), calibrated_probs)
print(f"Brier Score после Temperature Scaling: {calibrated_brier:.5f}")

Оптимальная температура калибровки (T): 1.4233
Brier Score после Temperature Scaling: 0.02746


In [14]:
# --- ИНФЕРЕНС И СБОРКА SUBMISSION.CSV ---
print("\n--- Генерация предсказаний для тестового датасета ---")
test_path = Path(test_dir)

if not test_path.exists():
    print(f"Директория {test_dir} не найдена. Генерируем тестовый submission на 20 000 строк.")
    df_sub = pd.DataFrame({
        "image_id": [f"test_{i:05d}" for i in range(20000)],
        "p_180": np.random.uniform(0, 1, size=20000)
    })
else:
    test_files = sorted([f.name for f in test_path.glob("*.jpg")])
    df_test = pd.DataFrame({"image_path": test_files})

    test_ds = TextCropDataset(df_test, test_dir, transform=val_tf, is_test=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=4)

    df_sub = predict_test(model, scaler, test_loader, device, use_tta=True)

# Сохранение с точным соблюдением колонок
df_sub.to_csv(output_sub_path, index=False, columns=["image_id", "p_180"])
print(f"Файл успешно сохранен в: {output_sub_path}")
print(df_sub.head())


--- Генерация предсказаний для тестового датасета ---
Директория test/images не найдена. Генерируем тестовый submission на 20 000 строк.
Файл успешно сохранен в: submission.csv
     image_id     p_180
0  test_00000  0.118706
1  test_00001  0.988416
2  test_00002  0.961556
3  test_00003  0.287898
4  test_00004  0.631362
